# Day 10 — Query understanding and transformations: change the question, measured

*Part of **15 Days of Modern, Enterprise-Relevant RAG from Scratch**. Lesson: [`docs/day-10/LEARN.md`](../docs/day-10/LEARN.md).*

Day 09 ended with four failures that start in the **question**, not in the documents. Two are about words: "How much time off do new dads get?" never meets "caregivers", and "Will Acme pay for wine with a business dinner?" never meets "alcohol". Two are about how Day 06's names rule reads a question: it refuses "What is the hotel limit per night in Tokyo?", although the travel policy has a row for any city not listed, and "My laptop might have a virus. Should I switch it off?", because it takes "Should", the first word of a second sentence, for a name.

**Query understanding** means reading the question before searching with it: which parts are values that must not change, which are names, how many sentences there are, and whether it is an exact value typed as a query. A **query transformation** changes what the retriever searches for. This notebook builds one of each from scratch and measures both:

1. **Query analysis**: sentences, values, names and places, and a route that leaves exact values alone.
2. **A reviewed vocabulary map** (query expansion): every entry tied to a recorded failure; the documents' words are appended to BM25's query and nothing is removed.
3. **The same 81 queries** against Day 09's accepted baseline, and **41 new held-out questions** written today (`questions-v2`).
4. **When not to rewrite**: a map matched word by word, one more entry that looks harmless, exact values, unanswerable questions, and a guard that any rewriter, a language model included, has to pass.
5. **The names rule as query analysis**: every sentence's first word skipped, and places typed. Then end to end.

**Every rule was committed before anything scored a held-out question.** [`evals/preregistration/day-10.md`](../evals/preregistration/day-10.md), the first commit on today's branch together with the new questions, holds the map, the rules, their dev values, what would be reported and what each decision needs. The notebook recomputes each rule on dev and stops if it gets another value.

Everything runs locally: no API key, no new packages and no new model (Day 04's embedding model, and Day 09's reranker, which is loaded only to reproduce Day 09's baseline), about a minute.

## 0. Setup

No new packages. The shared code in [`src/rag_from_scratch/`](../src/rag_from_scratch/) gains **Day 09's reranker** (`reranking.py`: `Reranker`, `rerank()` and `rerank_margin()`), because today reproduces Day 09's baseline, and **Day 07's `outcome()`** (in `evaluation.py`), because section 8 grades answers end to end. `tests/test_shared_code.py` checks both against the notebooks they came from.

In [1]:
import dataclasses
import hashlib
import json
import math
import platform
import re
import statistics
import sys
import time
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

assert sys.version_info >= (3, 11), "This series needs Python 3.11 or newer."
# huggingface_hub's progress bars warn when Jupyter widgets aren't installed; they still work as plain text.
warnings.filterwarnings("ignore", message="IProgress not found")

try:
    import numpy as np
    import onnxruntime as ort
    import tokenizers
except ImportError as error:
    raise ImportError("Day 10 uses Day 04's packages (numpy, onnxruntime, tokenizers, huggingface-hub). From the "
                      "repository root run: python -m pip install -r requirements.txt") from error


def find_repo_root() -> Path:
    # Walk up until the shared package is found, so the notebook works from the repo root or
    # from notebooks/. No hard-coded paths.
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "src" / "rag_from_scratch").is_dir():
            return folder
    raise FileNotFoundError("Could not find src/rag_from_scratch. Start Jupyter from inside the cloned repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))           # import the shared code without installing it

from rag_from_scratch.answering import (ABSTAIN, DIRECTORY, NAME, SENTENCE, ExtractiveReader,  # Day 06's pipeline
                                        Pipeline, Reply, answer_units, assemble, unknown_names)
from rag_from_scratch.bm25 import BM25Index, tokenize                    # Day 01's BM25
from rag_from_scratch.chunking import section_chunks                     # Day 03's chunker
from rag_from_scratch.embeddings import Embedder                         # Day 04's model
from rag_from_scratch.evaluation import (TIE, compare, first_relevant, hit, load_queries, measure,  # Day 07's harness
                                         outcome, sign_test, tie_margin, top_score, validate, wilson)
from rag_from_scratch.fusion import ALPHA, RRF_K, rrf, rrf_margin, weighted   # Day 08's fusion
from rag_from_scratch.index import VectorIndex, sync                     # Day 05's index
from rag_from_scratch.ingest import ingest                               # Day 02's pipeline
from rag_from_scratch.reranking import RERANKER, Reranker, rerank, rerank_margin   # Day 09's reranker (new in src/)

print(f"Python {platform.python_version()} | numpy {np.__version__} | onnxruntime {ort.__version__} | "
      f"tokenizers {tokenizers.__version__}")

Python 3.13.18 | numpy 2.4.6 | onnxruntime 1.31.0 | tokenizers 0.23.3


In [2]:
TOP_K = 3                # Day 07's definition: an ERROR is an answerable query with no labelled chunk in its top 3
K_VALUES = (1, 3, 5, 10)
EVALS = REPO_ROOT / "evals"
V1 = {"questions": EVALS / "questions-v1.jsonl", "manifest": EVALS / "manifest.json"}       # Day 07's 81 queries
V2 = {"questions": EVALS / "questions-v2.jsonl", "manifest": EVALS / "manifest-v2.json"}    # + today's 41 new ones
DAY09_BASELINE = EVALS / "baselines" / "day-09-retrieval.json"     # Day 09's accepted run: today's reference on v1
BASELINE_FILES = {1: EVALS / "baselines" / "day-10-retrieval.json",       # today's accepted runs, committed and
                  2: EVALS / "baselines" / "day-10-retrieval-v2.json"}    # reviewed: one per version of the questions
PREREGISTRATION = EVALS / "preregistration" / "day-10.md"          # today's rules, committed before any held-out run
OUT_DIR = REPO_ROOT / "outputs" / "day-10"                         # this run's results; outputs/ is git-ignored

# What evals/preregistration/day-10.md commits to. Sections 4 and 8 recompute each rule on the dev split and stop if
# the rule gives anything else.
PREREGISTERED = {"variant": "lexical",
                 "dev": {"no expansion": (46, 43, 0.9601), "lexical": (46, 44, 0.9746), "both": (46, 44, 0.9746)},
                 "names_dev": {"Day 06 names rule": (16, 16, 8, 8), "new names rule": (16, 16, 8, 8)}}
DEPTH = {"weighted": 3, "RRF": 20}   # Day 09's committed rerank depths (evals/preregistration/day-09.md)
DENSE_GATE = 0.62                    # Day 07's measured gate on the dense top score (evals/baselines/day-07-retrieval.json)
READER_THRESHOLD = 0.63              # Day 07's measured reader threshold (same file)

prereg = PREREGISTRATION.read_text(encoding="utf-8")
assert "**the rule chooses lexical.**" in prereg and "| lexical | 46 | 44 | 0.9746 |" in prereg \
    and "**the new rule gives the same, 16 of 16 and 8 of 8**" in prereg, "PREREGISTERED must match " \
    "evals/preregistration/day-10.md"

## 1. The same queries, the same snapshot, Day 09's six retrievers

A comparison is only fair if everything except the new step stays the same. So, before changing anything, this section loads Day 07's labels, checks the documents against their snapshot, rebuilds Day 09's six retrievers (BM25, dense, RRF, the weighted fusion, and both hybrids reranked) and checks that they **reproduce Day 09's accepted baseline**, query by query.

In [3]:
manifest = json.loads(V1["manifest"].read_text(encoding="utf-8"))
queries = load_queries(V1["questions"])
by_qid = {q.id: q for q in queries}
documents, _ = ingest(REPO_ROOT)
chunks = [c for d in documents for c in section_chunks(d)]
by_cid = {c.chunk_id: c for c in chunks}
problems = validate(queries, chunks, manifest)
v1_sha256 = hashlib.sha256(V1["questions"].read_bytes()).hexdigest()
if v1_sha256 != manifest["questions_sha256"]:
    problems.insert(0, f"{V1['questions'].name} changed but its manifest didn't")
print(f"{len(queries)} labelled queries (version {manifest['version']}) | {len(chunks)} chunks | label problems: "
      f"{len(problems)}" + "".join(f"\n  {p}" for p in problems))
assert not problems, "The labels don't match the documents: run the Day 07 notebook and review them first."

embedder = Embedder(REPO_ROOT)     # Day 04's model: downloaded once (133 MB), SHA-256 checked on every run
index = VectorIndex(embedder.model["repo"], embedder.revision, embedder.dimensions)
sync(index, chunks, embedder.embed)   # the snapshot's chunks, embedded now (Day 05's sync into an empty index)
rag = Pipeline(index, embedder)       # Day 06's retrieve(): as the user, permission filter before the ranking
bm25 = {user: BM25Index({c.chunk_id: c.text for c in chunks if set(c.allowed_groups) & groups})
        for user, groups in DIRECTORY.items()}
reranker = Reranker(REPO_ROOT)        # Day 09's cross-encoder (91 MB, SHA-256 checked), now from src/


def bm25_search(user: str, text: str) -> list[tuple[str, float]]:
    # Day 07's BM25: the user's own index, full ranking. Only chunks that share a word with the text come back.
    return [(h.id, h.score) for h in bm25[user].search(text, len(chunks))]


def dense_search(user: str, text: str) -> list[tuple[str, float]]:
    # Day 07's dense retrieval: every chunk the user may read, ranked by cosine similarity
    return [(r.id, r.score) for r in rag.retrieve(user, text, len(chunks))]


pair_scores = {}     # (question, chunk id) -> reranker logit: each pair is scored once


def reranked(question: str, ranked, n: int) -> list[tuple[str, float]]:
    # Day 09's rerank(): the hybrid's top n in the cross-encoder's order, then the rest in the hybrid's order
    top = [cid for cid, _ in ranked[:n]]
    missing = [cid for cid in top if (question, cid) not in pair_scores]
    for cid, score in zip(missing, reranker.scores(question, [by_cid[cid].text for cid in missing])):
        pair_scores[question, cid] = score
    return rerank(ranked, {cid: pair_scores[question, cid] for cid in top}, n)


DAY09_NAMES = ["BM25", "dense", "RRF", "weighted", "weighted+rerank", "RRF+rerank"]
rankings = {name: {} for name in DAY09_NAMES}     # retriever -> query id -> FULL ranking, (chunk_id, score) best first


def add_day09_rankings(qs, names=DAY09_NAMES) -> None:
    for q in qs:
        lexical, dense = bm25_search(q.user, q.question), dense_search(q.user, q.question)
        rankings["BM25"][q.id], rankings["dense"][q.id] = lexical, dense
        rankings["weighted"][q.id] = weighted(lexical, dense, ALPHA)
        if "RRF" in names:
            rankings["RRF"][q.id] = rrf([lexical, dense])
            rankings["weighted+rerank"][q.id] = reranked(q.question, rankings["weighted"][q.id], DEPTH["weighted"])
            rankings["RRF+rerank"][q.id] = reranked(q.question, rankings["RRF"][q.id], DEPTH["RRF"])


add_day09_rankings(queries)
SELECTIONS = {"questions": [q for q in queries if q.kind == "question" and q.answerable],
              "lookups": [q for q in queries if q.kind == "lookup"]}
print(f"Day 09's six retrievers on {len(queries)} queries: {len(SELECTIONS['questions'])} answerable questions, "
      f"{len(SELECTIONS['lookups'])} lookups")

81 labelled queries (version 1) | 44 chunks | label problems: 0
tokenizer.json       711,396 bytes   SHA-256 d241a60d5e8f... verified


onnx/model.onnx  133,093,490 bytes   SHA-256 828e1496d7fa... verified


tokenizer.json       711,396 bytes   SHA-256 d241a60d5e8f... verified


onnx/model.onnx   90,992,115 bytes   SHA-256 c623d0bcb99f... verified


Day 09's six retrievers on 81 queries: 31 answerable questions, 30 lookups


In [4]:
day09 = json.loads(DAY09_BASELINE.read_text(encoding="utf-8"))
DATASETS = {1: {"questions": V1["questions"].relative_to(REPO_ROOT).as_posix(), "version": manifest["version"],
                "sha256": v1_sha256, "snapshot": manifest["snapshot"]["fingerprint"]}}
SETTINGS = {   # Day 09's six retrievers exactly as Day 09 described them, so compare() checks the settings too
    "BM25": {"code": "rag_from_scratch.bm25 (Day 01)", "k1": bm25["alice"].k1, "b": bm25["alice"].b,
             "indexed_text": "chunk text with its Title > Section header",
             "permissions": "one index of each user's readable chunks (filter before the ranking)"},
    "dense": {"code": "rag_from_scratch.answering.Pipeline.retrieve (Day 06)", "model": embedder.model["repo"],
              "revision": embedder.revision, "query_instruction": True,
              "indexed_text": "chunk text with its Title > Section header",
              "permissions": "metadata filter before the ranking"},
    "RRF": {"code": "rrf() in the Day 08 notebook", "fuses": ["BM25", "dense"], "k": RRF_K,
            "inputs": "full rankings, each already permission-filtered", "ties": "chunk id"},
    "weighted": {"code": "weighted() in the Day 08 notebook", "fuses": ["BM25", "dense"], "alpha_dense": ALPHA,
                 "normalisation": "min-max per retriever and query; 0 from a retriever that didn't return the chunk",
                 "alpha_rule": "most dev queries in the top 3, then the higher dev MRR@10, then closest to 0.5",
                 "inputs": "full rankings, each already permission-filtered", "ties": "chunk id"}}
for base in ("weighted", "RRF"):
    SETTINGS[f"{base}+rerank"] = {"code": "rerank() in the Day 09 notebook", "reranks": base, "depth": DEPTH[base],
                                  "depth_rule": "most dev queries in the top 3, then the higher dev MRR@10, then the smaller N",
                                  "model": reranker.model["repo"], "revision": reranker.revision,
                                  "converted_from": reranker.model["converted_from"],
                                  "input": "question, then the chunk text with its Title > Section header; 512 tokens at most",
                                  "score": "the model's logit; ties by chunk id", "after_the_depth": "the hybrid's own order"}
RERANKED = {"weighted+rerank": "weighted", "RRF+rerank": "RRF"}


def entry(name: str, q) -> dict:
    # One query, one retriever, in the baseline's format: the rank of the first labelled chunk, how close that is to a
    # tie, the top 3 and the best score. RRF's margin is in its inputs (Day 08); a reranked one is Day 09's.
    ranked = rankings[name][q.id]
    if not q.answerable:
        margin = None
    elif name == "RRF":
        margin = rrf_margin([rankings["BM25"][q.id], rankings["dense"][q.id]], q.answer_chunks)
    elif name in RERANKED:
        margin = rerank_margin(rankings[RERANKED[name]][q.id], ranked, q.answer_chunks, DEPTH[RERANKED[name]])
    else:
        margin = tie_margin(ranked, q.answer_chunks)
    return {"first_relevant": first_relevant(ranked, q.answer_chunks) if q.answerable else None,
            "margin": margin, "top3": [cid for cid, _ in ranked[:TOP_K]], "top_score": top_score(ranked)}


def metrics_of(names, selections) -> dict:
    return {f"{kind}, {name}": {k: round(v, 4) if isinstance(v, float) else v
                                for k, v in measure(rankings[name], selection).items()}
            for kind, selection in selections.items() for name in names}


# Day 09's accepted file, with today's settings, ranks and metrics for its six retrievers in place of its own
reproduced = {**day09, "dataset": DATASETS[1], "retrievers": {name: SETTINGS[name] for name in DAY09_NAMES},
              "queries": {q.id: {name: entry(name, q) for name in DAY09_NAMES} for q in queries},
              "metrics": metrics_of(DAY09_NAMES, SELECTIONS)}
changes09, ties09 = compare(reproduced, day09)
same_top3 = all(reproduced["queries"][qid][n]["top3"] == day09["queries"][qid][n]["top3"] for qid in by_qid
                for n in DAY09_NAMES)
print(f"{', '.join(DAY09_NAMES)} against {DAY09_BASELINE.relative_to(REPO_ROOT).as_posix()}: "
      + (f"{len(changes09)} changes" + "".join(f"\n  {c}" for c in changes09) if changes09 else
         f"no changes ({len(queries)} queries x 6 retrievers: every rank, best score and metric; top 3 lists "
         f"{'identical' if same_top3 else 'DIFFERENT'})"))
for t in ties09:
    print(f"  near tie, not a change: {t}")

BM25, dense, RRF, weighted, weighted+rerank, RRF+rerank against evals/baselines/day-09-retrieval.json: no changes (81 queries x 6 retrievers: every rank, best score and metric; top 3 lists identical)


Same labels, same snapshot, same six rankings: every difference below comes from today's changes.

### 1.1 New held-out questions: `questions-v2`

Day 09 used the v1 test split to make two decisions, so its questions now count as used (`evals/README.md`, rules 4 and 8). Today's decisions need questions that nothing has been tuned on. [`evals/questions-v2.jsonl`](../evals/questions-v2.jsonl) keeps v1's 81 queries exactly as they were (its 27 used test questions become dev; each note says so) and adds **41 new test questions**, written today after the rules were frozen and committed with them, before any system ran on them. The pre-registration says what the author knew while writing them.

In [5]:
manifest2 = json.loads(V2["manifest"].read_text(encoding="utf-8"))
queries2 = load_queries(V2["questions"])
problems2 = validate(queries2, chunks, manifest2)
v2_sha256 = hashlib.sha256(V2["questions"].read_bytes()).hexdigest()
if v2_sha256 != manifest2["questions_sha256"]:
    problems2.insert(0, f"{V2['questions'].name} changed but its manifest didn't")
moved_to_dev = [q.id for q in queries if q.split == "test"]
# v2 keeps v1's 81 queries exactly (text, user, labels); only the used test questions changed split (and say so)
same81 = all(q2 == dataclasses.replace(by_qid[q2.id], split=q2.split, note=q2.note) for q2 in queries2[:81]) and \
    [q.id for q in queries2[:81] if q.split != by_qid[q.id].split] == moved_to_dev and \
    all(q.note.endswith("so dev from v2 on)") for q in queries2[:81] if q.id in moved_to_dev)
NEW = [q for q in queries2 if q.split == "test"]
by_qid.update({q.id: q for q in NEW})
print(f"{len(queries2)} labelled queries (version {manifest2['version']}): v1's {len(queries)} "
      f"({'identical, its ' + str(len(moved_to_dev)) + ' test questions now dev' if same81 else 'CHANGED'}) + "
      f"{len(NEW)} new test questions ({sum(q.answerable for q in NEW)} answerable, "
      f"{sum(not q.answerable for q in NEW)} not), same snapshot | label problems: {len(problems2)}"
      + "".join(f"\n  {p}" for p in problems2))
assert not problems2 and same81 and manifest2["snapshot"] == manifest["snapshot"]
assert {q.origin for q in NEW} == {"day10-new"} and all(q.kind == "question" for q in NEW)

122 labelled queries (version 2): v1's 81 (identical, its 27 test questions now dev) + 41 new test questions (29 answerable, 12 not), same snapshot | label problems: 0


## 2. Four failures that start in the question

In [6]:
DADS, WINE, TOKYO, SHOULD = (by_qid[qid] for qid in ("d01-01", "d07-a08", "d07-a01", "d07-a11"))


def rank(name: str, qid: str):
    return first_relevant(rankings[name][qid], by_qid[qid].answer_chunks)


for q in (DADS, WINE):
    answer = by_cid[q.answer_chunks[0]]
    shared = sorted(set(tokenize(q.question)) & set(tokenize(answer.text)))
    print(f"{q.id} ({q.split}) {q.question}\n  BM25's words for it:  {' '.join(tokenize(q.question))}\n"
          f"  shared with {answer.chunk_id}: {', '.join(shared) or 'none'}\n"
          f"  rank of the answer:  BM25 {rank('BM25', q.id) or '-'} | dense {rank('dense', q.id)} | "
          f"weighted fusion {rank('weighted', q.id)}\n")
for q in (TOKYO, SHOULD):
    results = rag.retrieve(q.user, q.question)     # the three chunks Day 06's pipeline would send
    sent = [r.id for r in results]
    print(f"{q.id} ({q.split}) {q.question}\n  Day 06's names rule: no source mentions "
          f"{unknown_names(q.question, assemble(q.question, results).sources)} -> refused, although the answer "
          f"{q.answer_chunks[0]} is source #{sent.index(q.answer_chunks[0]) + 1} of the 3 sent\n")

d01-01 (dev) How much time off do new dads get?
  BM25's words for it:  much time off new dads get
  shared with parental-leave-policy#leave-entitlement: none
  rank of the answer:  BM25 - | dense 3 | weighted fusion 3

d07-a08 (test) Will Acme pay for wine with a business dinner?
  BM25's words for it:  acme pay wine business dinner
  shared with expense-reimbursement-policy#meals: business
  rank of the answer:  BM25 6 | dense 4 | weighted fusion 3

d07-a01 (test) What is the hotel limit per night in Tokyo?
  Day 06's names rule: no source mentions ['Tokyo'] -> refused, although the answer travel-policy#daily-limits-by-city is source #1 of the 3 sent

d07-a11 (test) My laptop might have a virus. Should I switch it off?
  Day 06's names rule: no source mentions ['Should'] -> refused, although the answer security-incident-reporting#what-not-to-do is source #1 of the 3 sent



BM25 matches words. The new-dads question shares **no** word with its answer, so BM25 never returns the answer at all; dense retrieval finds it third, among three parental-leave chunks it scores within 0.06 of each other. The wine question shares only "business". And the names rule refuses two questions whose answer is the **first** source it was given.

## 3. Query understanding: what is in the question

Before changing a question, read it. `analyse()` splits it into sentences (Day 06's `SENTENCE` pattern) and finds three things:

- **Values**: numbers, amounts, emails, URLs and `#channels`. Whatever happens to the question, these must come through unchanged: "USD 40" decides whether a dinner needs a receipt.
- **Names**: Day 06's `NAME` pattern (capitalised words and acronyms), skipping the first word of **every** sentence, with places looked up first in a **gazetteer**, a list of place names and their types. This one is a stand-in chosen by an external rule, the G20's member countries and their capitals, so that no place is in it because a test question mentions it. A real one (GeoNames, for example) has millions of entries.
- **A route**: an exact value typed as the query (no `?`, at most 4 words, and a digit, `@`, `#` or `://`) is left alone. Day 04 showed that BM25 already finds those.

In [7]:
# A stand-in gazetteer (a list of place names with their types). Chosen by an external rule, the G20's member
# countries and their capitals, so that no place is here because a test question mentions it.
COUNTRIES = ["Argentina", "Australia", "Brazil", "Canada", "China", "France", "Germany", "India", "Indonesia", "Italy",
             "Japan", "Mexico", "Russia", "Saudi Arabia", "South Africa", "South Korea", "Turkey", "United Kingdom",
             "United States"]
CAPITALS = ["Buenos Aires", "Canberra", "Brasilia", "Ottawa", "Beijing", "Paris", "Berlin", "New Delhi", "Jakarta",
            "Rome", "Tokyo", "Mexico City", "Moscow", "Riyadh", "Pretoria", "Seoul", "Ankara", "London", "Washington"]
PLACES = {**{c: "country" for c in COUNTRIES}, **{c: "city" for c in CAPITALS}}
PLACE = re.compile(r"\b(?:" + "|".join(re.escape(p) for p in sorted(PLACES, key=len, reverse=True)) + r")\b")
VALUE = re.compile(r"https?://\S+|[\w.+-]+@[\w-]+(?:\.[\w-]+)+|#[\w-]+|USD \d[\d,]*(?:\.\d+)?|\d+(?:[.,:]\d+)*")


@dataclass(frozen=True)
class QueryAnalysis:
    question: str
    sentences: tuple[str, ...]
    values: tuple[str, ...]                    # numbers, amounts, emails, URLs, #channels: any rewrite must keep them
    names: tuple[tuple[str, str | None], ...]  # (name, place type or None): capitalised words, acronyms, places
    exact: bool                                # an exact value typed as the query: never expanded or rewritten


def question_names(question: str) -> tuple[tuple[str, str | None], ...]:
    # Day 06's NAME pattern, but skip the first word of EVERY sentence (Day 06 skipped only the question's first word),
    # and find gazetteer places first, as whole phrases ("New Delhi" is one name, not two)
    found = []
    for sentence in SENTENCE.split(question.strip()):
        places = [(m.start(), m.end(), m.group()) for m in PLACE.finditer(sentence)]
        for m in NAME.finditer(sentence):
            if m.start() == 0 or any(start <= m.start() < end for start, end, _ in places):
                continue                  # a sentence's first word is capitalised anyway; a place is handled whole
            found.append((m.group(), None))
        found += [(p, PLACES[p]) for start, _, p in places if start != 0]
    return tuple(found)


def analyse(question: str) -> QueryAnalysis:
    values = tuple(v.rstrip(".,!?") for v in VALUE.findall(question))
    exact = "?" not in question and len(question.split()) <= 4 and bool(re.search(r"\d|@|#|://", question))
    return QueryAnalysis(question, tuple(s for s in SENTENCE.split(question.strip()) if s), values,
                         question_names(question), exact)


for qid in ("d07-a11", "d07-a01", "d07-a02", "d01-03", "d04-v16", "d04-i01"):
    a = analyse(by_qid[qid].question)
    print(f"{qid:<9}{a.question:<56} sentences {len(a.sentences)} | values {list(a.values) or '-'} | names "
          f"{[n if t is None else f'{n} ({t})' for n, t in a.names] or '-'}" + (" | EXACT" if a.exact else ""))

d07-a11  My laptop might have a virus. Should I switch it off?    sentences 2 | values - | names -
d07-a01  What is the hotel limit per night in Tokyo?              sentences 1 | values - | names ['Tokyo (city)']
d07-a02  Do I need a receipt for a USD 40 dinner?                 sentences 1 | values ['USD 40'] | names ['USD']
d01-03   What is Google's remote work policy?                     sentences 1 | values - | names ['Google']
d04-v16  USD 1,000                                                sentences 1 | values ['USD 1,000'] | names - | EXACT
d04-i01  #it-help                                                 sentences 1 | values ['#it-help'] | names - | EXACT


"Should" is no longer a name, and Tokyo is a city. Day 06's pattern counts "USD" as an acronym: harmless for the names rule (every source with a price says USD), and the guard in section 7.3 checks it together with its value. Both lookups are exact: nothing will rewrite them.

## 4. A transformation: a reviewed vocabulary map

The simplest transformation aimed at a vocabulary gap is **query expansion**: add the documents' word next to the asker's. Search engines ship this as synonym lists (Azure AI Search's synonym maps, Elasticsearch's synonym filters). Three rules keep it honest:

1. **Every entry is tied to a gap someone recorded**, here a published diagnosis from Days 01 to 09. A map isn't a thesaurus.
2. **Entries match whole phrases** ("pay for", not "pay"), case-insensitive.
3. **Append, never replace**: the asker's words stay in the query.

Plus the abbreviations the documents define themselves ("Multi-factor authentication (MFA)"), harvested from the chunks, so nobody has to write those entries.

In [8]:
VOCABULARY = [   # (what the asker may say, what the documents say, where the gap was recorded)
    (("dad", "dads", "father", "fathers", "mum", "mums", "mom", "moms", "mother", "mothers"), ("caregivers",),
     "d01-01 (dev): dads vs caregivers; with the singular, a synonym and the counterparts"),
    (("time off",), ("leave",), "d01-01 (dev): time off vs leave"),
    (("wine", "beer", "spirits"), ("alcohol",), "d07-a08 (v1 test, used): wine vs alcohol"),
    (("pay for",), ("reimbursed",), "d07-a08 (v1 test, used): pay for vs reimbursed"),
    (("work from home", "working from home", "wfh"), ("remotely", "remote"),
     "d07-a03 (v1 test, used): work from home vs work remotely"),
    (("virus",), ("infected",), "d07-a11 (v1 test, used): virus vs infected"),
    (("switch off", "switch it off", "shut down", "shut it down", "power off"), ("turn off",),
     "d07-a11 (v1 test, used): switch off vs turn off"),
    (("locked",), ("locks", "lockout"), "d01-02 (dev): locked vs locks (BM25 has no stemming)"),
]
DEFINED = re.compile(r"\b([A-Z][a-z]+(?:[- ][a-z]+){0,4}) \(([A-Z]{2,})\)")    # "Long form (ABBR)"


def defined_abbreviations(chunks) -> list[tuple]:
    # The abbreviations the documents define themselves, both ways: no one has to write these entries
    pairs = sorted({m.groups() for c in chunks for m in DEFINED.finditer(c.text)})
    return [entry for long, short in pairs for entry in (
        ((short.lower(),), (long.lower(),), f"defined in the documents: {long} ({short})"),
        ((long.lower(),), (short.lower(),), f"defined in the documents: {long} ({short})"))]


ENTRIES = VOCABULARY + defined_abbreviations(chunks)


def contains(text: str, phrase: str) -> bool:
    # A whole word or phrase, case-insensitive: "dad" is in "new dad?" but not in "Baghdad"
    return re.search(rf"(?<!\w){re.escape(phrase.lower())}(?!\w)", text.lower()) is not None


@dataclass(frozen=True)
class Expansion:
    text: str                                      # what BM25 searches for: the question, then the added terms
    matched: tuple[tuple[str, tuple[str, ...]], ...]   # (phrase found in the question, terms it added)
    route: str                                     # "expanded", "no match" or "exact value: not expanded"


def expand(question: str, entries=ENTRIES) -> Expansion:
    # Query expansion with a reviewed map: APPEND the documents' words; never remove or replace the asker's
    if analyse(question).exact:
        return Expansion(question, (), "exact value: not expanded")
    matched = [(next(p for p in phrases if contains(question, p)), terms) for phrases, terms, _ in entries
               if any(contains(question, p) for p in phrases)]
    added = list(dict.fromkeys(t for _, terms in matched for t in terms))
    return Expansion(f"{question} {' '.join(added)}" if added else question, tuple(matched),
                     "expanded" if added else "no match")


print(f"{len(VOCABULARY)} reviewed entries + {len(ENTRIES) - len(VOCABULARY)} harvested from the documents' own "
      f"definitions ({', '.join(e[2].split(': ', 1)[1] for e in ENTRIES[len(VOCABULARY)::2])})\n")
routes = Counter(expand(q.question).route for q in queries)
print(f"v1's 81 queries: {routes['expanded']} expanded, {routes['no match']} without a match, "
      f"{routes['exact value: not expanded']} exact values left alone")
for q in queries:
    e = expand(q.question)
    if e.matched:
        why = "; ".join(f"{p} -> {', '.join(t)}" for p, t in e.matched)
        print(f"  {q.id:<9}{q.split:<5} {q.question:<56} + {' '.join(e.text[len(q.question):].split())}   ({why})")

8 reviewed entries + 2 harvested from the documents' own definitions (Multi-factor authentication (MFA))

v1's 81 queries: 6 expanded, 45 without a match, 30 exact values left alone
  d01-01   dev   How much time off do new dads get?                       + caregivers leave   (dads -> caregivers; time off -> leave)
  d01-02   dev   My account is locked, what now?                          + locks lockout   (locked -> locks, lockout)
  d07-a03  test  How many days a week can I work from home?               + remotely remote   (work from home -> remotely, remote)
  d07-a08  test  Will Acme pay for wine with a business dinner?           + alcohol reimbursed   (wine -> alcohol; pay for -> reimbursed)
  d07-a11  test  My laptop might have a virus. Should I switch it off?    + infected turn off   (virus -> infected; switch it off -> turn off)
  d07-u12  test  How much does Acme pay for a home internet connection?   + reimbursed   (pay for -> reimbursed)


The map touches 6 of v1's 81 queries; the 30 lookups are routed around it. Three of the six are v1 test questions that entries were **written from** (d07-a03, d07-a08, d07-a11), so whatever happens to them is in-sample by construction: a description, not evidence. One unanswerable question is touched too ("pay for" in the home-internet question); section 7.2 looks at it.

### 4.1 Where the expansion goes: the pre-registered rule, on dev

The expanded text can go to BM25 alone, or to both retrievers. The rule, committed in [`evals/preregistration/day-10.md`](../evals/preregistration/day-10.md): *on dev (16 answerable dev questions and 30 lookups), the variant with the most queries in the weighted fusion's top 3, then the most first, then the higher MRR@10; a tie goes to "lexical", which changes one input instead of two.*

In [9]:
DEV = [q for q in queries if q.split == "dev" and q.answerable]     # the queries we may tune on (16 + 30 lookups)


def with_map(q, variant: str = "lexical", entries=ENTRIES):
    # BM25 searches the expanded text; dense retrieval the original question ("lexical") or the expanded one ("both")
    text = expand(q.question, entries).text
    lexical = bm25_search(q.user, text)
    dense = dense_search(q.user, text) if variant == "both" else rankings["dense"][q.id]
    return lexical, dense, weighted(lexical, dense, ALPHA)


def summary(fused, selection) -> tuple:
    m = measure(fused, selection)
    return m["hit@3"], m["hit@1"], round(m["MRR@10"], 4)


# The rule (pre-registered; dev only): most dev queries in the top 3, then most first, then the higher MRR@10; a tie
# goes to "lexical", which changes one input instead of two
variant_sweep = {"no expansion": summary(rankings["weighted"], DEV)}
variant_sweep.update({v: summary({q.id: with_map(q, v)[2] for q in DEV}, DEV) for v in ("lexical", "both")})
VARIANT = max(("lexical", "both"), key=lambda v: (*variant_sweep[v], v == "lexical"))
print(f"{'dev only (46)':<16}{'top 3':>7}{'first':>7}{'MRR@10':>9}   changed on dev")
for v, counts in variant_sweep.items():
    moved = [] if v == "no expansion" else [
        f"{q.id}: {first_relevant(rankings['weighted'][q.id], q.answer_chunks)} -> "
        f"{first_relevant(with_map(q, v)[2], q.answer_chunks)}" for q in DEV
        if first_relevant(with_map(q, v)[2], q.answer_chunks) != first_relevant(rankings["weighted"][q.id], q.answer_chunks)]
    print(f"{v:<16}{counts[0]:>7}{counts[1]:>7}{counts[2]:>9.4f}   {', '.join(moved)}")
print(f"-> {VARIANT}")
assert VARIANT == PREREGISTERED["variant"] and variant_sweep == PREREGISTERED["dev"], \
    "the rule doesn't give the pre-registered variant or values: stop and find out why"

MAPPED = {"BM25+map": "BM25", "weighted+map": "weighted"}    # today's two retrievers, and what each is compared with


def add_map_rankings(qs) -> None:
    for q in qs:
        lexical, _, fused = with_map(q, VARIANT)
        rankings.setdefault("BM25+map", {})[q.id] = lexical
        rankings.setdefault("weighted+map", {})[q.id] = fused


add_map_rankings(queries)
NAMES = DAY09_NAMES + list(MAPPED)

dev only (46)     top 3  first   MRR@10   changed on dev
no expansion         46     43   0.9601   
lexical              46     44   0.9746   d01-01: 3 -> 1


both                 46     44   0.9746   d01-01: 3 -> 1
-> lexical


A tie on every count: on dev only new dads moves, and the same way in both variants. The tie-break picks **lexical**: dense retrieval keeps the original question. Today's two retrievers are **`BM25+map`** (BM25 on the expanded question) and **`weighted+map`** (Day 08's weighted fusion of `BM25+map` and dense retrieval).

## 5. On v1's 81 queries, against Day 09's baseline

In [10]:
def table(names, selections) -> None:
    width = max(len(f"{kind}, {name}") for kind in selections for name in names) + 2
    print(f"{'':<{width}}{'n':>3}" + "".join(f"{'hit@' + str(k):>7}" for k in K_VALUES) + f"{'MRR@10':>8}"
          "   hit@3 rate, 95% interval")
    for kind, selection in selections.items():
        for name in names:
            m = measure(rankings[name], selection)
            low, high = wilson(m["hit@3"], m["n"])
            print(f"{kind + ', ' + name:<{width}}{m['n']:>3}" + "".join(f"{m['hit@' + str(k)]:>7}" for k in K_VALUES)
                  + f"{m['MRR@10']:>8.2f}   {m['hit@3'] / m['n']:.0%} ({low:.0%} to {high:.0%})")


table(["BM25", "BM25+map", "weighted", "weighted+map"], SELECTIONS)

                           n  hit@1  hit@3  hit@5 hit@10  MRR@10   hit@3 rate, 95% interval
questions, BM25           31     24     29     29     30    0.85   94% (79% to 98%)
questions, BM25+map       31     27     31     31     31    0.92   100% (89% to 100%)
questions, weighted       31     27     31     31     31    0.92   100% (89% to 100%)
questions, weighted+map   31     28     31     31     31    0.95   100% (89% to 100%)
lookups, BM25             30     30     30     30     30    1.00   100% (89% to 100%)
lookups, BM25+map         30     30     30     30     30    1.00   100% (89% to 100%)
lookups, weighted         30     30     30     30     30    1.00   100% (89% to 100%)
lookups, weighted+map     30     30     30     30     30    1.00   100% (89% to 100%)


In the top 3 nothing can improve for the weighted fusion: it already had all 31 answerable questions there. BM25 on its own goes from 29 to 31. First place: the weighted fusion gains one (28 vs 27).

### 5.1 Query by query

Each new retriever against its own base **as recorded in `evals/baselines/day-09-retrieval.json`**, pair by pair, with the sign test, in three parts: the 16 answerable dev questions, the 15 answerable v1 test questions (used, and in-sample for the map), and the 30 lookups.

In [11]:
def accepted_rank(name: str, qid: str, accepted=None):
    # A retriever recorded in an accepted baseline: the rank (and margin) recorded there. Otherwise: this run's.
    if accepted is not None and name in accepted["retrievers"] and qid in accepted["queries"]:
        return accepted["queries"][qid][name]["first_relevant"], accepted["queries"][qid][name]["margin"]
    e = entry(name, by_qid[qid]) if name not in MAPPED else {
        "first_relevant": rank(name, qid), "margin": tie_margin(rankings[name][qid], by_qid[qid].answer_chunks)}
    return e["first_relevant"], e["margin"]


PARTS = {"dev questions": [q for q in SELECTIONS["questions"] if q.split == "dev"],
         "v1 test questions (used)": [q for q in SELECTIONS["questions"] if q.split == "test"],
         "lookups": SELECTIONS["lookups"]}
PAIRED, NEAR = {}, {}


def show_paired(a: str, b: str, k: int, parts, accepted=None) -> None:
    for part, selection in parts.items():
        ranks = {name: {q.id: accepted_rank(name, q.id, accepted) for q in selection} for name in (a, b)}
        found = {name: {qid for qid, (r, _) in ranks[name].items() if r is not None and r <= k} for name in (a, b)}
        # CPU rounding could explain a difference only if one side is a near tie (< TIE from a neighbour) sitting right
        # at the cut-off (rank k or k + 1), where one swap moves it across: those are set apart, not counted
        near = {qid for qid in found[a] ^ found[b]
                if any(ranks[n][qid][0] in (k, k + 1) and (ranks[n][qid][1] or 1) < TIE for n in (a, b))}
        groups = {"both": found[a] & found[b], f"only {a}": found[a] - found[b] - near,
                  f"only {b}": found[b] - found[a] - near, "neither": {q.id for q in selection} - found[a] - found[b]}
        PAIRED[a, b, k, part] = {label: sorted(ids) for label, ids in groups.items()}
        NEAR[a, b, k, part] = sorted(near)
        p = sign_test(len(groups[f"only {a}"]), len(groups[f"only {b}"]))
        where = "first" if k == 1 else f"in the top {k}"
        print(f"{a} vs {b}, {part} ({len(selection)}), labelled chunk {where}: "
              + " | ".join(f"{label} {len(ids)}" for label, ids in groups.items()) + f" | sign test p = {p:.2g}"
              + (f" | near ties set apart: {', '.join(sorted(near))}" if near else ""))
        for label in (f"only {a}", f"only {b}"):
            if groups[label]:
                print(f"    {label:<22} " + ", ".join(sorted(groups[label])))


for k in (1, 3):
    for a, b in MAPPED.items():
        show_paired(a, b, k, PARTS, accepted=day09)
    print()

BM25+map vs BM25, dev questions (16), labelled chunk first: both 13 | only BM25+map 1 | only BM25 0 | neither 2 | sign test p = 1
    only BM25+map          d01-01
BM25+map vs BM25, v1 test questions (used) (15), labelled chunk first: both 11 | only BM25+map 2 | only BM25 0 | neither 2 | sign test p = 0.5
    only BM25+map          d07-a03, d07-a08
BM25+map vs BM25, lookups (30), labelled chunk first: both 30 | only BM25+map 0 | only BM25 0 | neither 0 | sign test p = 1
weighted+map vs weighted, dev questions (16), labelled chunk first: both 13 | only weighted+map 1 | only weighted 0 | neither 2 | sign test p = 1
    only weighted+map      d01-01
weighted+map vs weighted, v1 test questions (used) (15), labelled chunk first: both 14 | only weighted+map 0 | only weighted 0 | neither 1 | sign test p = 1
weighted+map vs weighted, lookups (30), labelled chunk first: both 30 | only weighted+map 0 | only weighted 0 | neither 0 | sign test p = 1

BM25+map vs BM25, dev questions (16), labelled 

Every query whose rank moved, with the reason:

In [12]:
# Every v1 query whose rank changed when the map was added, with the reason (written after reading each one).
# The assert fails if the set of moves changes, so a new move can't go unexplained.
WHY_MOVED = {
    ("BM25+map", "d01-01"): "BM25 shared no word with the entitlement; caregivers and leave do",
    ("BM25+map", "d07-a03"): "remotely matches Eligibility ('work remotely up to three days')",
    ("BM25+map", "d07-a08"): "alcohol and reimbursed match Meals ('Alcohol is not reimbursed')",
    ("weighted+map", "d01-01"): "BM25's new first place lifts the entitlement above the two leave chunks dense prefers",
    ("weighted+map", "d07-a08"): "BM25 now puts Meals first, but dense still prefers Scope and has 0.7 of the weight",
}
moves = [(name, q) for name, base in MAPPED.items() for q in queries if q.answerable
         and rank(name, q.id) != rank(base, q.id)]
assert {(name, q.id) for name, q in moves} == set(WHY_MOVED), "every move needs a reason, and only moves"
print(f"{'':<14}{'query':<9}{'split':<6}{'rank':>10}   added to BM25's query     why")
for name, q in moves:
    print(f"{name:<14}{q.id:<9}{q.split:<6}{f'{rank(MAPPED[name], q.id)} -> {rank(name, q.id)}':>10}   "
          f"{' '.join(expand(q.question).text[len(q.question):].split()):<26}{WHY_MOVED[name, q.id]}")

              query    split       rank   added to BM25's query     why
BM25+map      d01-01   dev    None -> 1   caregivers leave          BM25 shared no word with the entitlement; caregivers and leave do
BM25+map      d07-a03  test      2 -> 1   remotely remote           remotely matches Eligibility ('work remotely up to three days')
BM25+map      d07-a08  test      6 -> 1   alcohol reimbursed        alcohol and reimbursed match Meals ('Alcohol is not reimbursed')
weighted+map  d01-01   dev       3 -> 1   caregivers leave          BM25's new first place lifts the entitlement above the two leave chunks dense prefers
weighted+map  d07-a08  test      3 -> 2   alcohol reimbursed        BM25 now puts Meals first, but dense still prefers Scope and has 0.7 of the weight


Five moves, all gains, no losses; none the sign test can tell from luck (p = 1 and 0.5), and two of the five on questions the map was written from. **Read the wine row**: BM25 now puts Meals first ("Alcohol is not reimbursed"), but dense retrieval still prefers the Scope chunk and has 0.7 of the weight, so the fusion moves wine from 3rd to 2nd, not to 1st. Fixing one retriever's vocabulary doesn't fix the fusion.

## 6. The held-out test: 41 new questions

The comparison the pre-registration made: `weighted+map` against the weighted fusion, and `BM25+map` against BM25, on the 29 answerable new questions. Nothing about these questions was used to set anything.

In [13]:
V2_NAMES = ["BM25", "dense", "weighted", "BM25+map", "weighted+map"]   # what the pre-registration reports on v2
add_day09_rankings(NEW, names=V2_NAMES)   # the new questions: nothing about them was used to set anything today
add_map_rankings(NEW)
HELD_OUT = {"v2 test questions": [q for q in NEW if q.answerable]}
routes2 = Counter(expand(q.question).route for q in NEW)
print(f"v2's {len(NEW)} new test questions: {routes2['expanded']} expanded, {routes2['no match']} without a match")
for q in NEW:
    e = expand(q.question)
    if e.matched:
        print(f"  {q.id:<9}{q.question:<60} + {' '.join(e.text[len(q.question):].split())}")
WHY_MOVED.update({("BM25+map", "d10-a17"): "remotely matches Eligibility, as for d07-a03; the weighted fusion had it first"})
moves2 = [(name, q) for name, base in MAPPED.items() for q in NEW if q.answerable and rank(name, q.id) != rank(base, q.id)]
assert {(name, q.id) for name, q in moves2} == {k for k in WHY_MOVED if k[1].startswith("d10")}
print()
table(V2_NAMES, HELD_OUT)
print()
for k in (1, 3):
    for a, b in MAPPED.items():
        show_paired(a, b, k, HELD_OUT)

v2's 41 new test questions: 1 expanded, 40 without a match
  d10-a17  Do warehouse staff get to work from home?                    + remotely remote

                                   n  hit@1  hit@3  hit@5 hit@10  MRR@10   hit@3 rate, 95% interval
v2 test questions, BM25           29     20     26     26     27    0.78   90% (74% to 96%)
v2 test questions, dense          29     23     27     28     28    0.86   93% (78% to 98%)
v2 test questions, weighted       29     24     27     28     29    0.89   93% (78% to 98%)
v2 test questions, BM25+map       29     21     26     26     27    0.80   90% (74% to 96%)
v2 test questions, weighted+map   29     24     27     28     29    0.89   93% (78% to 98%)

BM25+map vs BM25, v2 test questions (29), labelled chunk first: both 20 | only BM25+map 1 | only BM25 0 | neither 8 | sign test p = 1
    only BM25+map          d10-a17
weighted+map vs weighted, v2 test questions (29), labelled chunk first: both 24 | only weighted+map 0 | only weighted 0 

**The map met one of the 41 new questions**, "Do warehouse staff get to work from home?", and the weighted fusion already had its answer first. Against the weighted fusion: no gain and no loss, at first place or in the top 3. BM25 alone gains that one first place.

The new questions bring their own gaps (part-time vs reduced hours, two-factor vs multi-factor, type it wrong vs failed sign-in attempts, food vs meals), and the map has none of them. **A map fixes the gaps someone recorded, and only those.** On this corpus dense retrieval already bridges most paraphrases, which is why the fused retriever loses so little to them.

### 6.1 Every miss

In [14]:
DIAGNOSES = {(e["retriever"], e["id"]): (e["category"], e["why"]) for e in day09["errors"]}   # Day 09's, on v1
NEW_DIAGNOSES = {   # every v2 test miss (written after reading each one); the assert below fails if the set changes
    ("BM25", "d10-a10"): ("vocabulary", "no word in common: part-time, baby, born vs phased schedule, reduced hours, child"),
    ("BM25+map", "d10-a10"): ("vocabulary", "the map has no entry for part-time or baby: nothing to add"),
    ("BM25", "d10-a12"): ("vocabulary", "only 'password' is shared, and five chunks mention it; type it wrong vs failed "
                                        "sign-in attempts"),
    ("BM25+map", "d10-a12"): ("vocabulary", "the map has no entry for 'type it wrong': nothing to add"),
    ("weighted", "d10-a12"): ("vocabulary", "both retrievers rank Passwords and Resetting your password above Account "
                                            "lockout (dense #3, BM25 #6)"),
    ("weighted+map", "d10-a12"): ("vocabulary", "as for the weighted fusion: the map added nothing"),
    ("BM25", "d10-a24"): ("other section", "How to report shares no word with the question; Safety while travelling "
                                           "(phone, stolen, trip) comes first and says report it, not to whom"),
    ("BM25+map", "d10-a24"): ("other section", "as for BM25: the map added nothing"),
    ("dense", "d10-a24"): ("other section", "How to report never mentions a phone or a trip: 5th, behind Safety while "
                                            "travelling and three other incident sections"),
    ("weighted", "d10-a24"): ("other section", "the question names the event, the answer names the procedure: neither "
                                               "retriever has How to report in its top 3 (6th)"),
    ("weighted+map", "d10-a24"): ("other section", "as for the weighted fusion: the map added nothing"),
    ("dense", "d10-a28"): ("buried sentence", "the answer is the last of three sentences in a chunk about MFA; "
                                              "'someone from IT' pulls the IT-help chunks up (18th)"),
}
DIAGNOSES.update(NEW_DIAGNOSES)
misses = [(name, q) for name in NAMES for q in queries if q.answerable and not hit(rankings[name][q.id], q.answer_chunks, TOP_K)]
misses2 = [(name, q) for name in V2_NAMES for q in NEW if q.answerable and not hit(rankings[name][q.id], q.answer_chunks, TOP_K)]
print(f"v1, misses in the top 3: " + ", ".join(f"{name} {sum(n == name for n, _ in misses)}" for name in NAMES))
print(f"v2 test, misses in the top 3: " + ", ".join(f"{name} {sum(n == name for n, _ in misses2)}" for name in V2_NAMES))
assert {(name, q.id) for name, q in misses2 + [(n, q) for n, q in misses if n in MAPPED]} == set(NEW_DIAGNOSES), \
    "every miss needs a diagnosis, and only misses"
for name, q in misses2:
    category, why = DIAGNOSES[name, q.id]
    print(f"  {name:<14}{q.id:<9}#{rank(name, q.id) or '-':<4}{category:<16}{q.question}\n{'':<45}{why}")

v1, misses in the top 3: BM25 2, dense 13, RRF 1, weighted 0, weighted+rerank 0, RRF+rerank 2, BM25+map 0, weighted+map 0
v2 test, misses in the top 3: BM25 3, dense 2, weighted 2, BM25+map 3, weighted+map 2
  BM25          d10-a10  #-   vocabulary      Can I come back part-time after my baby is born?
                                             no word in common: part-time, baby, born vs phased schedule, reduced hours, child
  BM25          d10-a12  #6   vocabulary      What happens if I type my password wrong too many times?
                                             only 'password' is shared, and five chunks mention it; type it wrong vs failed sign-in attempts
  BM25          d10-a24  #-   other section   My phone was stolen on a business trip. Who should I tell?
                                             How to report shares no word with the question; Safety while travelling (phone, stolen, trip) comes first and says report it, not to whom
  dense         d10-a24  #5   other se

The weighted fusion misses 2 of the 29 new answerable questions: a vocabulary gap the map doesn't cover ("type my password wrong too many times" vs "failed sign-in attempts"), and Day 03's lost-laptop pattern again, the event in one section and the procedure in another ("My phone was stolen on a business trip. Who should I tell?"). Day 11 retrieves surrounding sections.

## 7. When not to rewrite

### 7.1 Word by word, and one more entry

Two variants of the map, both pre-registered as descriptions (they decide nothing): **(a)** the same entries matched word by word, as an unreviewed word list would; **(b)** the map plus one entry that looks harmless, "dinner, lunch, breakfast → meals".

In [15]:
# (a) The same map, matched word by word instead of as phrases: what an unreviewed word list does
WORD_ENTRIES = [(tuple(sorted({w for p in phrases for w in p.split()})), terms, why) for phrases, terms, why in ENTRIES]
# (b) The map plus one more entry that looks harmless: a dinner is a meal
MEALS_ENTRIES = ENTRIES + [(("dinner", "lunch", "breakfast"), ("meals",), "not recorded: added for this demo")]
ALL_PARTS = {**PARTS, **HELD_OUT}


def effect(entries, parts=ALL_PARTS) -> dict:
    # Against the weighted fusion without any map: queries touched, and gains / losses at first place and in the top 3
    result = {}
    for part, selection in parts.items():
        touched = [q for q in selection if expand(q.question, entries).matched]
        fused = {q.id: with_map(q, VARIANT, entries)[2] for q in selection}
        gains, losses = [], []
        for k in (1, 3):
            for q in selection:
                before, after = hit(rankings["weighted"][q.id], q.answer_chunks, k), hit(fused[q.id], q.answer_chunks, k)
                (gains if after > before else losses if after < before else []).append(f"{q.id}@{k}")
        result[part] = (len(touched), gains, losses)
    return result


EFFECTS = {"reviewed map (phrases)": effect(ENTRIES), "(a) the same map, word by word": effect(WORD_ENTRIES),
           "(b) + dinner, lunch, breakfast -> meals": effect(MEALS_ENTRIES)}
for label, per_part in EFFECTS.items():
    print(label)
    for part, (touched, gains, losses) in per_part.items():
        print(f"    {part:<26} touches {touched:>2} of {len(ALL_PARTS[part]):<3} gains {', '.join(gains) or '-':<24} "
              f"losses {', '.join(losses) or '-'}")

reviewed map (phrases)
    dev questions              touches  2 of 16  gains d01-01@1                 losses -
    v1 test questions (used)   touches  3 of 15  gains -                        losses -
    lookups                    touches  0 of 30  gains -                        losses -
    v2 test questions          touches  1 of 29  gains -                        losses -
(a) the same map, word by word
    dev questions              touches  7 of 16  gains d01-01@1                 losses -
    v1 test questions (used)   touches  6 of 15  gains -                        losses d07-a02@1
    lookups                    touches  0 of 30  gains -                        losses -
    v2 test questions          touches 11 of 29  gains d10-a05@1                losses d10-a28@1
(b) + dinner, lunch, breakfast -> meals
    dev questions              touches  2 of 16  gains d01-01@1                 losses -
    v1 test questions (used)   touches  4 of 15  gains -                        losses d0

Every query either variant moves:

In [16]:
for q in [q for q in queries + NEW if q.answerable]:
    for label, entries in (("word by word", WORD_ENTRIES), ("+ meals", MEALS_ENTRIES)):
        fused = with_map(q, VARIANT, entries)[2]
        if first_relevant(fused, q.answer_chunks) != rank("weighted", q.id):
            print(f"{label:<13}{q.id:<9}{q.question:<58} adds: {' '.join(expand(q.question, entries).text[len(q.question):].split()):<34}"
                  f"rank {rank('weighted', q.id)} -> {first_relevant(fused, q.answer_chunks)} (top: {fused[0][0]})")

word by word d01-01   How much time off do new dads get?                         adds: caregivers leave turn off         rank 3 -> 1 (top: parental-leave-policy#leave-entitlement)
+ meals      d01-01   How much time off do new dads get?                         adds: caregivers leave                  rank 3 -> 1 (top: parental-leave-policy#leave-entitlement)
word by word d07-a02  Do I need a receipt for a USD 40 dinner?                   adds: reimbursed                        rank 1 -> 2 (top: expense-reimbursement-policy#meals)
+ meals      d07-a02  Do I need a receipt for a USD 40 dinner?                   adds: meals                             rank 1 -> 2 (top: expense-reimbursement-policy#meals)
word by word d07-a08  Will Acme pay for wine with a business dinner?             adds: alcohol reimbursed                rank 3 -> 2 (top: expense-reimbursement-policy#scope)
+ meals      d07-a08  Will Acme pay for wine with a business dinner?             adds: alcohol reimbursed meals    

- **Word by word, the map touches 11 of the 29 new questions instead of 1.** "for" (from "pay for") adds *reimbursed* to any question with "for", "it" (from "switch it off") adds *turn off*, and "time" in "one-time" adds *leave*: "Can I give my one-time code to someone from IT?" gets *leave remotely remote turn off* and loses its first place. It also gains one (*reimbursed* helps the expense-deadline question) and loses the receipt question, so the word list looks fine on dev and fails elsewhere.
- **"Dinner → meals" is true and still hurts.** The receipt question ("Do I need a receipt for a USD 40 dinner?") needs the general receipt rule, not the meal limit; with *meals* added, the Meals chunk goes first. It doesn't fix wine either: dense retrieval still has the weight.

Both are why the map matches phrases and why every entry needs a recorded reason. An entry that is true in general can still be wrong for a question.

### 7.2 Exact values and unanswerable questions

In [17]:
# (c) Routing, and (d) what the map does to questions the documents don't answer
lookups_left = sum(expand(q.question).route == "exact value: not expanded" for q in SELECTIONS["lookups"])
print(f"(c) lookups left unexpanded by the routing: {lookups_left} of {len(SELECTIONS['lookups'])}")
unanswerable = [q for q in queries + NEW if q.kind == "question" and not q.answerable]
changed_top = [q for q in unanswerable if rankings["weighted+map"][q.id][0][0] != rankings["weighted"][q.id][0][0]]
print(f"(d) unanswerable questions ({len(unanswerable)}) whose first chunk changes with the map: {len(changed_top)}")
for q in changed_top:
    print(f"    {q.id:<9}{q.question:<52} {rankings['weighted'][q.id][0][0]} -> {rankings['weighted+map'][q.id][0][0]}")
touched_unanswerable = [q.id for q in unanswerable if expand(q.question).matched]
print(f"    touched by the map at all: {', '.join(touched_unanswerable) or 'none'}; Day 07's gate reads dense retrieval, "
      "which still sees the original question, so no refusal can change")
# (e) A rewrite changes WHAT is searched for, never WHO searches: the user, and so the permission filter, come from the
# login (Day 05), and every retriever filters before ranking (Day 08). Checked on every retriever and every query:
leaks = [(name, q.id) for name in rankings for q in queries + NEW if q.id in rankings[name]
         and set(q.restricted_chunks) & {cid for cid, _ in rankings[name][q.id][:10]}]
print(f"(e) restricted chunks in any top 10 ({len(rankings)} retrievers on v1's queries, {len(V2_NAMES)} on the new ones): "
      f"{len(leaks)}")

(c) lookups left unexpanded by the routing: 30 of 30
(d) unanswerable questions (32) whose first chunk changes with the map: 0
    touched by the map at all: d07-u12; Day 07's gate reads dense retrieval, which still sees the original question, so no refusal can change
(e) restricted chunks in any top 10 (8 retrievers on v1's queries, 5 on the new ones): 0


The routing keeps every lookup away from the map. The one unanswerable question the map touches ("How much does Acme pay for a home internet connection?", *reimbursed* added) keeps its first chunk, and no unanswerable question changes its first chunk. Day 07's gate reads dense retrieval's top score, and dense retrieval still sees the original question, so expansion on BM25's side can't change a refusal. That is one more reason the rule's choice, expanding only BM25's query, is the safer one. And a rewrite changes *what* is searched for, never *who* searches: the user, and with it the permission filter, come from the login, and every retriever filters before ranking, so no restricted chunk reaches any top 10.

### 7.3 Any rewriter needs a guard

A language model can rewrite a question far more freely than a map: fix its grammar, resolve "it", split it in two, or add the documents' vocabulary for words no one recorded. It can also drop or change the part that decides the answer. `check_rewrite()` lists what a rewrite broke, using `analyse()`: a value lost or added, a name lost or added, an empty rewrite. A rewrite with any problem is discarded and the original question is searched. Exact values are never rewritten.

The rewrites below are **written by hand to show one failure each; none is a model's output.** The new-dads one is modelled on what `gpt-6-luna` did in Day 06's live run: its *answer* assumed new dads are secondary caregivers (6 weeks), although the policy is gender-neutral. Here that assumption is written as a *rewrite*.

In [18]:
def check_rewrite(original: str, rewritten: str) -> list[str]:
    # What a rewrite (from a model, a template or a person) broke. Any problem: discard it and search the original.
    if not rewritten.strip():
        return ["empty"]
    before, after = analyse(original), analyse(rewritten)
    # names that are part of a value ("USD" in "USD 40") are checked with the value
    names_before = {n for n, _ in before.names if not any(n in v.split() for v in before.values)}
    names_after = {n for n, _ in after.names if not any(n in v.split() for v in after.values)}
    return ([f"drops {v}" for v in before.values if v not in after.values]
            + [f"adds {v}" for v in after.values if v not in before.values]
            + [f"drops the name {n}" for n in sorted(names_before - names_after)]
            + [f"adds the name {n}" for n in sorted(names_after - names_before)])


REWRITES = [   # written by hand to show one failure each: NOT a model's output
    ("d07-a02", "Do I need a receipt for a dinner?", "drops the value that decides the answer"),
    ("d07-a01", "What is the hotel limit per night in Japan?", "swaps the name for a broader one"),
    ("d04-v18", "What is the hotel limit of USD 220 per night?", "turns an exact value into a question"),
    ("d01-01", "How much leave do secondary caregivers get?", "commits to one reading: dads can be primary caregivers"),
    ("d07-a08", "Will Acme reimburse alcohol with a business dinner?", "keeps every value and name"),
]
GUARD = {}
for qid, rewrite, shows in REWRITES:
    original = by_qid[qid].question
    route = "exact value: never rewritten" if analyse(original).exact else None
    problems = check_rewrite(original, rewrite)
    GUARD[qid] = route or ("discarded: " + "; ".join(problems) if problems else "kept")
    print(f"{original:<44} -> {rewrite:<52}\n    {shows:<58} guard: {GUARD[qid]}")

Do I need a receipt for a USD 40 dinner?     -> Do I need a receipt for a dinner?                   
    drops the value that decides the answer                    guard: discarded: drops USD 40
What is the hotel limit per night in Tokyo?  -> What is the hotel limit per night in Japan?         
    swaps the name for a broader one                           guard: discarded: drops the name Tokyo; adds the name Japan
USD 220                                      -> What is the hotel limit of USD 220 per night?       
    turns an exact value into a question                       guard: exact value: never rewritten
How much time off do new dads get?           -> How much leave do secondary caregivers get?         
    commits to one reading: dads can be primary caregivers     guard: kept
Will Acme pay for wine with a business dinner? -> Will Acme reimburse alcohol with a business dinner? 
    keeps every value and name                                 guard: kept


The guard catches what it can see: a value dropped, a name swapped, an exact value turned into a question. **It keeps the new-dads rewrite**, because no value or name changed, only the meaning: a dad can be a primary caregiver (16 weeks). A guard checks form, not meaning. Expansion doesn't have this failure: it adds *caregivers* without choosing primary or secondary, and leaves the reading to the documents.

**No language-model rewrite is run in this notebook** (that was pre-registered). It would cost a model call per question, couldn't run in CI, and would have to beat the map under these same rules, on questions nothing was tuned on, to earn its place. Exercise 2 shows where it would plug in: behind `check_rewrite()`.

## 8. The names rule, as query analysis

Day 06's reader refuses when the question names something no source mentions. Day 07 measured that rule and found it refused two answerable test questions. The new rule changes two things, one per recorded failure: **every sentence's first word is skipped** ("Should"), and **a gazetteer place counts as known if a source mentions its type** ("Use the Other cities row for any city not listed"). Everything else is Day 06's. Both rules are compared behind Day 07's measured dense gate (0.62), reading the three sources Day 06's pipeline would send, on dev, on the used v1 test split, and on the new questions.

In [19]:
def unknown_names_v2(question: str, sources) -> list[str]:
    # Day 06's names rule with the two changes: every sentence's first word skipped, and a gazetteer place counts as
    # known if a source mentions its TYPE ("Use the Other cities row for any city not listed")
    text = " ".join(s.text for s in sources)
    unknown = [name for name, kind in question_names(question)
               if not contains(text, name) and not (kind and any(contains(text, w) for w in TYPE_WORDS[kind]))]
    return list(dict.fromkeys(unknown))


TYPE_WORDS = {"city": ("city", "cities"), "country": ("country", "countries")}
NAMES_RULES = {"Day 06 names rule": unknown_names, "new names rule": unknown_names_v2}
natural = [q for q in queries + NEW if q.kind == "question"]          # gates are for questions, not pasted values
SPLITS = {"dev": [q for q in natural if q.split == "dev"], "v1 test (used)": [q for q in natural if q.split == "test"
          and q.id.startswith("d07")], "v2 test": [q for q in natural if q.id.startswith("d10")]}
sent = {q.id: rag.retrieve(q.user, q.question) for q in natural}     # what Day 06's pipeline would send: dense top 3
found = {(label, q.id): tuple(rule(q.question, assemble(q.question, sent[q.id]).sources))
         for label, rule in NAMES_RULES.items() for q in natural}


def refuses(label: str, q) -> bool:
    return sent[q.id][0].score < DENSE_GATE or bool(found[label, q.id])     # Day 07's gate, then the names rule


NAMES_TABLE = {}
print(f"{'dense gate 0.62, then':<24}" + "".join(f"{split:>28}" for split in SPLITS))
print(f"{'':<24}" + "".join(f"{'kept / refused':>28}" for _ in SPLITS))
for label in NAMES_RULES:
    cells = []
    for split, part in SPLITS.items():
        kept = sum(q.answerable and not refuses(label, q) for q in part)
        refused = sum(not q.answerable and refuses(label, q) for q in part)
        NAMES_TABLE[label, split] = (kept, sum(q.answerable for q in part), refused, sum(not q.answerable for q in part))
        cells.append(f"{kept} of {NAMES_TABLE[label, split][1]} / {refused} of {NAMES_TABLE[label, split][3]}")
    print(f"{label:<24}" + "".join(f"{c:>28}" for c in cells))
assert {label: NAMES_TABLE[label, "dev"] for label in NAMES_RULES} == PREREGISTERED["names_dev"], \
    "the names rules don't give the pre-registered dev values: stop and find out why"
DIFFERENT = [q for q in natural if refuses("Day 06 names rule", q) != refuses("new names rule", q)]
print(f"\nquestions whose decision differs ({len(DIFFERENT)}):")
for q in DIFFERENT:
    print(f"  {q.id:<9}{'answerable' if q.answerable else 'not answerable':<16}{q.question:<72} unknown names: "
          f"Day 06 {list(found['Day 06 names rule', q.id])} -> new {list(found['new names rule', q.id])}")
named = [q for q in SPLITS["v2 test"] if found["Day 06 names rule", q.id] or found["new names rule", q.id]]
print(f"\nv2 test questions with an unknown name under either rule ({len(named)}):")
for q in named:
    print(f"  {q.id:<9}{'answerable' if q.answerable else 'not answerable':<16}{q.question:<72} Day 06 "
          f"{list(found['Day 06 names rule', q.id])} | new {list(found['new names rule', q.id])}")

dense gate 0.62, then                            dev              v1 test (used)                     v2 test
                                      kept / refused              kept / refused              kept / refused
Day 06 names rule                  16 of 16 / 8 of 8          13 of 15 / 6 of 12          22 of 29 / 6 of 12
new names rule                     16 of 16 / 8 of 8          15 of 15 / 6 of 12          26 of 29 / 5 of 12

questions whose decision differs (7):
  d07-a01  answerable      What is the hotel limit per night in Tokyo?                              unknown names: Day 06 ['Tokyo'] -> new []
  d07-a11  answerable      My laptop might have a virus. Should I switch it off?                    unknown names: Day 06 ['Should'] -> new []
  d10-a01  answerable      What's the hotel budget per night in Berlin?                             unknown names: Day 06 ['Berlin'] -> new []
  d10-a02  answerable      How much can I spend on food per day in Paris?                        

On dev both rules give exactly the pre-registered values (16 of 16 kept, 8 of 8 refused). On the used v1 test split the new rule keeps Tokyo and the virus question, as it was written to. On the **new** questions it keeps **26 of 29 answerable questions instead of 22**: Berlin and Paris are cities, and two questions whose second sentence starts with "Do" or "Who" are no longer refused for a name that isn't one. Toronto isn't in the gazetteer, so it is still refused; Italy is a country, but no source mentions countries, so the Italy question is still refused, which is right.

**And it refuses one unanswerable question fewer (5 of 12 instead of 6).** "I'm moving to a new team. Does my remote work arrangement carry over?" was refused by Day 06's rule because it took "Does" for a name. That refusal was right by accident. Fixing a bug removed it.

### 8.1 End to end

A refusal that is lifted hands the question to the next stage. Day 06's reader with the names rule as a setting (its other steps copied unchanged, and checked against Day 06's reader), behind Day 07's measured gate and reader threshold, graded with Day 07's `outcome()`:

In [20]:
class NamesReader(ExtractiveReader):
    """Day 06's extractive reader with the names rule as a setting. The rest of generate() is Day 06's, unchanged."""

    def __init__(self, embedder, names_rule=unknown_names, **settings):
        super().__init__(embedder, **settings)
        self.names_rule = names_rule

    def generate(self, context):
        if not context.sources:
            return Reply(ABSTAIN, self.name, "no sources")
        missing = self.names_rule(context.question, context.sources)
        if missing:                                    # the question is about something the sources never mention
            return Reply(ABSTAIN, self.name, f"no source mentions {', '.join(missing)}")
        ranked = self.scored_units(context)
        best_score, best_source, _ = ranked[0]
        if best_score < self.min_score:                # nothing in the sources is close enough
            return Reply(ABSTAIN, self.name, f"best sentence scores {best_score:.2f} < {self.min_score}")
        picked = [u for score, s, u in ranked if s.id == best_source.id and score >= best_score - self.margin]
        order = answer_units(best_source)
        picked = sorted(picked[:self.max_sentences], key=order.index)        # in the document's order
        text = " ".join(f"{u.rstrip('.')} [{best_source.id}]." for u in picked)
        return Reply(text, self.name, f"best sentence scores {best_score:.2f} >= {self.min_score}")


pipeline = Pipeline(index, embedder, min_retrieval_score=DENSE_GATE)          # Day 07's measured gate
# First, the copy is faithful: with Day 06's rule it answers every question exactly as Day 06's reader does
same_reader = all(pipeline.answer(q.user, q.question, ExtractiveReader(embedder, min_score=READER_THRESHOLD)) ==
                  pipeline.answer(q.user, q.question, NamesReader(embedder, min_score=READER_THRESHOLD))
                  for q in natural if q.split == "dev")
print(f"NamesReader with Day 06's rule = Day 06's ExtractiveReader on the 24 dev questions: {same_reader}")
answers = {label: {q.id: pipeline.answer(q.user, q.question, NamesReader(embedder, rule, min_score=READER_THRESHOLD))
                   for q in natural} for label, rule in NAMES_RULES.items()}
outcomes = {label: {q.id: outcome(answers[label][q.id], q) for q in natural} for label in NAMES_RULES}
print(f"\n{'gate 0.62, reader 0.63':<24}{'split':<16}{'answerable: correct / wrong / refused':<40}"
      "not answerable: refused / answered")
for label in NAMES_RULES:
    for split, part in SPLITS.items():
        a = Counter(outcomes[label][q.id] for q in part if q.answerable)
        u = Counter(outcomes[label][q.id] for q in part if not q.answerable)
        answerable = f"{a['correct']} / {a['wrong']} / {a['refused']}  (of {sum(a.values())})"
        print(f"{label:<24}{split:<16}{answerable:<40}{u['refused']} / {u['answered']}  (of {sum(u.values())})")
CHANGED = [q for q in natural if outcomes["Day 06 names rule"][q.id] != outcomes["new names rule"][q.id]]
print(f"\nquestions whose outcome changes ({len(CHANGED)}):")
for q in CHANGED:
    a = answers["new names rule"][q.id]
    print(f"  {q.id:<9}{q.question:<72}{outcomes['Day 06 names rule'][q.id]} -> {outcomes['new names rule'][q.id]}\n"
          f"{'':<11}{a.text}")

NamesReader with Day 06's rule = Day 06's ExtractiveReader on the 24 dev questions: True



gate 0.62, reader 0.63  split           answerable: correct / wrong / refused   not answerable: refused / answered
Day 06 names rule       dev             14 / 2 / 0  (of 16)                     8 / 0  (of 8)
Day 06 names rule       v1 test (used)  12 / 1 / 2  (of 15)                     7 / 5  (of 12)
Day 06 names rule       v2 test         19 / 3 / 7  (of 29)                     6 / 6  (of 12)
new names rule          dev             14 / 2 / 0  (of 16)                     8 / 0  (of 8)
new names rule          v1 test (used)  14 / 1 / 0  (of 15)                     7 / 5  (of 12)
new names rule          v2 test         21 / 5 / 3  (of 29)                     5 / 7  (of 12)

questions whose outcome changes (7):
  d07-a01  What is the hotel limit per night in Tokyo?                             refused -> correct
           City: London | Hotel per night: USD 250 | Meals per day: USD 75 [travel-policy#daily-limits-by-city]. City: Singapore | Hotel per night: USD 220 | Meals per day: USD

Seven outcomes change, and every one of them is a question the new rule stopped refusing. Four become correct: Tokyo, the virus question, Berlin and "register the trip" (the hotel answers contain the right row, USD 180, next to rows that don't apply: the extractive reader copies the rows closest to the question; the grade is a containment check, not a judgement). **Three become wrong answers:** Paris gets London's row, the stolen phone gets "report it straight away" from the travel policy instead of whom to tell, and the team-move question gets a confident, cited sentence that doesn't answer it.

**A refusal can hide a failure downstream.** Lifting it moves the question to the next stage, which has failures of its own. Day 12 grades answers; Day 11 retrieves the section next to the one that matched.

## 9. What it costs

In [21]:
def timed(work, sample, rounds: int = 3) -> tuple[float, float]:
    samples = []
    for _ in range(rounds):
        for q in sample:
            started = time.perf_counter()
            work(q)
            samples.append((time.perf_counter() - started) * 1000)
    samples.sort()
    return statistics.median(samples), samples[max(0, math.ceil(0.95 * len(samples)) - 1)]


SAMPLE = (queries + NEW)[::4]     # 31 queries from every split
timings = {"expand the question (map + routing)": timed(lambda q: expand(q.question), SAMPLE),
           "BM25 (full ranking)": timed(lambda q: bm25_search(q.user, q.question), SAMPLE),
           "dense (embed the question + search)": timed(lambda q: dense_search(q.user, q.question), SAMPLE)}
for step, (median, p95) in timings.items():
    print(f"{step:<38} median {median:9.3f} ms   p95 {p95:9.3f} ms")

expand the question (map + routing)    median     0.089 ms   p95     0.139 ms
BM25 (full ranking)                    median     0.024 ms   p95     0.047 ms
dense (embed the question + search)    median    12.346 ms   p95    19.704 ms


The map and the routing take a small fraction of a millisecond per question, against 10 to 12 ms to embed it on this 2-vCPU machine. A language-model rewrite would add a model call per question, typically far longer than everything else here; Day 13 measures the whole pipeline.

## 10. Decisions

Both decisions were written down, with what each needs, before any held-out question was scored (`evals/preregistration/day-10.md`):

In [22]:
held_out = HELD_OUT["v2 test questions"]
lost = sorted({qid for k in (1, 3) for qid in PAIRED["weighted+map", "weighted", k, "v2 test questions"]["only weighted"]})
gained = sorted({qid for k in (1, 3) for qid in PAIRED["weighted+map", "weighted", k, "v2 test questions"]["only weighted+map"]})
D1 = not lost and bool(gained)
print(f"D1, the map in the default retriever: on the {len(held_out)} answerable v2 test questions it gains "
      f"{', '.join(gained) or 'nothing'} and loses {', '.join(lost) or 'nothing'} (first place or the top 3) -> "
      + ("adopted: later days build on weighted+map" if D1 else
         "not adopted: the weighted fusion stays the default; the map is recorded as measured"))

old, new = NAMES_TABLE["Day 06 names rule", "v2 test"], NAMES_TABLE["new names rule", "v2 test"]
D2 = new[0] >= old[0] and new[2] >= old[2] and (new[0] > old[0] or new[2] > old[2])
print(f"D2, the new names rule: on the {len(SPLITS['v2 test'])} v2 test questions it keeps {new[0]} of {new[1]} answerable "
      f"(Day 06's rule: {old[0]}) and refuses {new[2]} of {new[3]} unanswerable (Day 06's rule: {old[2]}) -> "
      + ("adopted (recorded for later days; answering.py unchanged)" if D2 else "recorded as measured, not adopted"))

D1, the map in the default retriever: on the 29 answerable v2 test questions it gains nothing and loses nothing (first place or the top 3) -> not adopted: the weighted fusion stays the default; the map is recorded as measured
D2, the new names rule: on the 41 v2 test questions it keeps 26 of 29 answerable (Day 06's rule: 22) and refuses 5 of 12 unanswerable (Day 06's rule: 6) -> recorded as measured, not adopted


**Neither is adopted.** The map gained nothing on the new questions: it met one, whose answer was already first. It didn't lose anything either, so "not adopted" means "no evidence it helps beyond the questions it was written from", not "harmful". The new names rule kept four more answerable questions and let one more unanswerable question through, and the bar set in advance was "loses nothing". The weighted fusion stays the default retriever, Day 06's names rule stays in `answering.py`, and both of today's changes are recorded with their numbers.

**The new test split is now used, too.** It decided D1 and D2, so under rule 4 its 41 questions count as used: a later decision that needs held-out evidence needs new questions again. Each decision on a held-out set spends it, which is a reason to write larger test sets and make fewer, better-specified decisions on them.

## 11. Save the runs, compare with the accepted baselines

Two run files, because a baseline belongs to one version of the questions: `outputs/day-10/day-10-retrieval.json` (v1's 81 queries, Day 09's six retrievers unchanged plus today's two, comparable with Day 09's file) and `outputs/day-10/day-10-retrieval-v2.json` (v2's 122 queries, the five retrievers the pre-registration reports on, the reference for later days). The accepted copies are committed in [`evals/baselines/`](../evals/baselines/). Both record the names-rule outcomes and the two decisions.

In [23]:
for name in MAPPED:
    SETTINGS[name] = {"code": "expand() in the Day 10 notebook", "expands": "BM25's query only (rule on dev)",
                      "map": [[list(phrases), list(terms)] for phrases, terms, _ in ENTRIES],
                      "matching": "whole phrases, case-insensitive; the documents' terms are appended, nothing removed",
                      "routing": "an exact value (no ?, at most 4 words, a digit, @, # or ://) is not expanded"}
SETTINGS["BM25+map"].update({"searches": "BM25 (Day 01, the user's own index) on the expanded question"})
SETTINGS["weighted+map"].update({"fuses": ["BM25+map", "dense"], "alpha_dense": ALPHA,
                                 "dense_query": "the original question"})
THRESHOLDS = {
    "preregistration": PREREGISTRATION.relative_to(REPO_ROOT).as_posix(),
    "names_rule": {"gate": f"Day 07's measured dense gate {DENSE_GATE} in front",
                   "sources": "dense retrieval's top 3, assembled (what Day 06's pipeline sends)",
                   "gazetteer": "G20 member countries and their capitals; a place is known if a source mentions its type",
                   "outcomes": {f"{label}, {split}": dict(zip(("answerable kept", "answerable", "unanswerable refused",
                                                               "unanswerable"), counts))
                                for (label, split), counts in NAMES_TABLE.items()}},
    "day09": "see evals/baselines/day-09-retrieval.json (reranker gate -3.6, not adopted)",
    "decisions": {"D1 weighted+map as the default retriever": "adopted" if D1 else "not adopted",
                  "D2 the new names rule": "adopted" if D2 else "not adopted"}}
DATASETS[2] = {"questions": V2["questions"].relative_to(REPO_ROOT).as_posix(), "version": manifest2["version"],
               "sha256": v2_sha256, "snapshot": manifest2["snapshot"]["fingerprint"]}
RUNS = {1: (queries, NAMES, SELECTIONS, misses), 2: (queries2, V2_NAMES, {
    "questions": [q for q in queries2 if q.kind == "question" and q.answerable],
    "lookups": [q for q in queries2 if q.kind == "lookup"]}, [(n, q) for n, q in misses if n in V2_NAMES] + misses2)}
OUT_DIR.mkdir(parents=True, exist_ok=True)
compared = {}
for version, (qs, names, selections, missed) in RUNS.items():
    run = {"name": f"Day 10 retrieval baseline (questions v{version})", "dataset": DATASETS[version],
           "retrievers": {name: SETTINGS[name] for name in names}, "top_k": TOP_K,
           "metrics": metrics_of(names, selections),
           "queries": {q.id: {name: entry(name, q) if name not in MAPPED else {
               "first_relevant": rank(name, q.id) if q.answerable else None,
               "margin": tie_margin(rankings[name][q.id], q.answer_chunks) if q.answerable else None,
               "top3": [cid for cid, _ in rankings[name][q.id][:TOP_K]], "top_score": top_score(rankings[name][q.id])}
               for name in names} for q in qs},
           "errors": [{"retriever": name, "id": q.id, "first_relevant": rank(name, q.id),
                       "category": DIAGNOSES[name, q.id][0], "why": DIAGNOSES[name, q.id][1]} for name, q in missed],
           "thresholds": THRESHOLDS}
    run_file = OUT_DIR / BASELINE_FILES[version].name
    run_file.write_text(json.dumps(run, indent=1, ensure_ascii=False) + "\n", encoding="utf-8", newline="\n")
    print(f"this run: {run_file.relative_to(REPO_ROOT).as_posix()} ({len(qs)} queries x {len(names)} retrievers, "
          f"{len(run['errors'])} errors)")
    accepted_file = BASELINE_FILES[version]
    if accepted_file.is_file():
        changes, ties = compare(run, json.loads(accepted_file.read_text(encoding="utf-8")))
        compared[version] = (changes, ties)
        print(f"compared with {accepted_file.relative_to(REPO_ROOT).as_posix()}: "
              + (f"{len(changes)} changes" + "".join(f"\n  {c}" for c in changes) if changes else
                 f"no changes ({len(qs)} queries x {len(names)} retrievers and their metrics)"))
        for t in ties:
            print(f"  near tie, not a change: {t}")
    else:
        compared[version] = (None, [])
        print(f"no accepted baseline yet: review {run_file.name}, then copy it to "
              f"{accepted_file.relative_to(REPO_ROOT).as_posix()} and commit it")
fragile = sorted((e["margin"], qid, name) for qid, per in json.loads((OUT_DIR / BASELINE_FILES[2].name).read_text(
    encoding="utf-8"))["queries"].items() for name, e in per.items() if name in MAPPED and e["margin"] is not None)[:3]
print("closest calls for today's two retrievers: " + ", ".join(f"{qid} {name} ({m})" for m, qid, name in fragile))

this run: outputs/day-10/day-10-retrieval.json (81 queries x 8 retrievers, 18 errors)
compared with evals/baselines/day-10-retrieval.json: no changes (81 queries x 8 retrievers and their metrics)
this run: outputs/day-10/day-10-retrieval-v2.json (122 queries x 5 retrievers, 27 errors)
compared with evals/baselines/day-10-retrieval-v2.json: no changes (122 queries x 5 retrievers and their metrics)
closest calls for today's two retrievers: d10-a28 weighted+map (0.0014), d10-a24 weighted+map (0.0026), d10-a02 weighted+map (0.0064)


The closest calls are on the new questions, all above the 0.001 that would make them near ties: the one-time-code question, where the weighted fusion's first two chunks are 0.0014 apart, then the stolen phone and Paris.

## 12. Sanity checks

Each check pins a result quoted in this notebook or in LEARN.md. Scores are compared with a tolerance; ids, ranks and counts exactly. Timings are checked only for their shape.

In [24]:
def counts(name: str, selection) -> tuple:
    m = measure(rankings[name], selection)
    return m["hit@1"], m["hit@3"], m["hit@5"], m["hit@10"]


# The same inputs as Day 09 (section 1), and the new questions
assert problems == [] and changes09 == [] and same_top3 and len(queries) == 81 and len(chunks) == 44
assert problems2 == [] and same81 and len(queries2) == 122 and len(NEW) == 41 and sum(q.answerable for q in NEW) == 29
assert manifest2["counts"]["dev"] == 81 and manifest2["counts"]["test"] == 41

# Four failures, and what the analysis sees (sections 2 and 3)
assert not set(tokenize(DADS.question)) & set(tokenize(by_cid[DADS.answer_chunks[0]].text))
assert set(tokenize(WINE.question)) & set(tokenize(by_cid[WINE.answer_chunks[0]].text)) == {"business"}
assert [unknown_names(q.question, assemble(q.question, rag.retrieve(q.user, q.question)).sources)
        for q in (TOKYO, SHOULD)] == [["Tokyo"], ["Should"]]
assert analyse(SHOULD.question).names == () and len(analyse(SHOULD.question).sentences) == 2
assert analyse(TOKYO.question).names == (("Tokyo", "city"),) and analyse("Is New Delhi covered?").names == (("New Delhi", "city"),)
assert [q.id for q in queries2 if analyse(q.question).exact] == [q.id for q in queries2 if q.kind == "lookup"]
assert analyse("Do I need a receipt for a USD 40 dinner?").values == ("USD 40",)

# The map (section 4) and the pre-registered rule
assert len(VOCABULARY) == 8 and len(ENTRIES) == 10 and routes == Counter({"no match": 45, "exact value: not expanded": 30,
                                                                          "expanded": 6})
assert [q.id for q in queries if expand(q.question).matched] == ["d01-01", "d01-02", "d07-a03", "d07-a08", "d07-a11", "d07-u12"]
assert expand("Is MFA needed?").text == "Is MFA needed? multi-factor authentication"
assert expand("How did Baghdad do?").route == "no match"                  # a whole word: "dad" isn't in "Baghdad"
assert VARIANT == "lexical" and rankings["weighted+map"]["d04-v01"] == rankings["weighted"]["d04-v01"]
assert all(rankings["BM25+map"][q.id] == rankings["BM25"][q.id] for q in queries + NEW if not expand(q.question).matched)

# v1, against Day 09's baseline (section 5)
assert counts("weighted+map", SELECTIONS["questions"]) == (28, 31, 31, 31) and counts("weighted", SELECTIONS["questions"]) == (27, 31, 31, 31)
assert counts("BM25+map", SELECTIONS["questions"]) == (27, 31, 31, 31) and counts("BM25", SELECTIONS["questions"]) == (24, 29, 29, 30)
assert all(counts(n, SELECTIONS["lookups"]) == (30, 30, 30, 30) for n in MAPPED)
assert PAIRED["weighted+map", "weighted", 1, "dev questions"]["only weighted+map"] == ["d01-01"]
assert all(not PAIRED["weighted+map", "weighted", k, p]["only weighted"] for k in (1, 3) for p in PARTS)   # no loss on v1
assert PAIRED["BM25+map", "BM25", 1, "v1 test questions (used)"]["only BM25+map"] == ["d07-a03", "d07-a08"]
assert len(moves) == 5 and rank("BM25+map", "d07-a08") == 1 and rank("weighted+map", "d07-a08") == 2
assert rank("weighted+map", "d01-01") == 1 and rank("weighted", "d01-01") == 3

# The held-out questions (section 6)
assert routes2 == Counter({"no match": 40, "expanded": 1}) and [q.id for q in NEW if expand(q.question).matched] == ["d10-a17"]
assert counts("weighted", HELD_OUT["v2 test questions"]) == counts("weighted+map", HELD_OUT["v2 test questions"]) == (24, 27, 28, 29)
assert counts("BM25", HELD_OUT["v2 test questions"]) == (20, 26, 26, 27) and counts("BM25+map", HELD_OUT["v2 test questions"]) == (21, 26, 26, 27)
assert counts("dense", HELD_OUT["v2 test questions"]) == (23, 27, 28, 28)
assert all(not PAIRED["weighted+map", "weighted", k, "v2 test questions"][g] for k in (1, 3)
           for g in ("only weighted+map", "only weighted"))
assert PAIRED["BM25+map", "BM25", 1, "v2 test questions"]["only BM25+map"] == ["d10-a17"]
assert len(misses) == 18 and len(misses2) == 12 and {q.id for n, q in misses2 if n == "weighted"} == {"d10-a12", "d10-a24"}

# When not to rewrite (section 7)
assert EFFECTS["(a) the same map, word by word"]["v1 test questions (used)"][2] == ["d07-a02@1"]
assert EFFECTS["(a) the same map, word by word"]["v2 test questions"][1:] == (["d10-a05@1"], ["d10-a28@1"])
assert EFFECTS["(b) + dinner, lunch, breakfast -> meals"]["v1 test questions (used)"][2] == ["d07-a02@1"]
assert all(not losses for _, _, losses in EFFECTS["reviewed map (phrases)"].values())
assert lookups_left == 30 and changed_top == [] and touched_unanswerable == ["d07-u12"] and leaks == []
assert set(rankings["RRF"]) == set(by_qid) - {q.id for q in NEW}       # nothing unreported was run on the new questions
assert GUARD == {"d07-a02": "discarded: drops USD 40", "d07-a01": "discarded: drops the name Tokyo; adds the name Japan",
                 "d04-v18": "exact value: never rewritten", "d01-01": "kept", "d07-a08": "kept"}

# The names rule (section 8) and end to end
assert NAMES_TABLE["Day 06 names rule", "v1 test (used)"] == (13, 15, 6, 12) == tuple(
    day09["thresholds"]["reranker_gate"]["outcomes"]["Day 07: dense gate 0.62 + names rule, test"].values())
assert NAMES_TABLE["new names rule", "v1 test (used)"] == (15, 15, 6, 12)
assert NAMES_TABLE["Day 06 names rule", "v2 test"] == (22, 29, 6, 12) and NAMES_TABLE["new names rule", "v2 test"] == (26, 29, 5, 12)
assert [q.id for q in DIFFERENT] == ["d07-a01", "d07-a11", "d10-a01", "d10-a02", "d10-a23", "d10-a24", "d10-u12"]
assert found["new names rule", "d10-a03"] == ("Toronto",) and found["new names rule", "d10-u03"] == ("Italy",)
assert same_reader and [q.id for q in CHANGED] == [q.id for q in DIFFERENT]
assert [outcomes["new names rule"][qid] for qid in ("d07-a01", "d10-a02", "d10-a24", "d10-u12")] == \
    ["correct", "wrong", "wrong", "answered"]

# Cost (section 9): the shape only, with a wide margin
assert timings["expand the question (map + routing)"][0] < timings["dense (embed the question + search)"][0] / 10

# Decisions (section 10) and the runs (section 11)
assert not D1 and not D2 and lost == [] and gained == []
saved = {v: json.loads((OUT_DIR / BASELINE_FILES[v].name).read_text(encoding="utf-8")) for v in (1, 2)}
assert saved[1]["dataset"] == day09["dataset"] and len(saved[1]["errors"]) == 18 and len(saved[2]["errors"]) == 27
assert all(saved[1]["queries"][qid][n] == reproduced["queries"][qid][n] for qid in day09["queries"] for n in DAY09_NAMES)
assert all(saved[2]["queries"][qid][n] == saved[1]["queries"][qid][n] for qid in day09["queries"] for n in V2_NAMES)
for version in (1, 2):
    changes, ties = compared[version]
    assert changes is None or changes == [], changes        # near ties (CPU rounding) are allowed
    if BASELINE_FILES[version].is_file():                   # and the same top 3 lists as the accepted run
        accepted = json.loads(BASELINE_FILES[version].read_text(encoding="utf-8"))
        assert all(saved[version]["queries"][qid][n]["top3"] == accepted["queries"][qid][n]["top3"]
                   for qid in accepted["queries"] for n in accepted["retrievers"])

print("All Day 10 sanity checks passed.")

All Day 10 sanity checks passed.


## 13. Exercises

Each takes 5–15 minutes. Restart and rerun the notebook after changing anything. Both test splits have now been used to decide, so treat everything here as dev.

1. **Add an entry for a gap you saw today.** Add "two-factor → multi-factor" to `VOCABULARY` (with its reason) and rerun. Which queries move? Why can't the 41 new questions tell you whether it helps any more, and what would?
2. **Plug in a language-model rewrite, behind the guard.** Write `rewrite(question) -> str` with Day 06's `ChatModel` (copy it from the Day 06 notebook, section 8; it needs `RAG_CHAT_*` in `.env`, and costs a little per call). Search with the rewrite only when `check_rewrite()` finds no problem. Run it on the dev questions first: does it beat the map under section 4.1's rule? What does it do to "new dads"?
3. **Multi-query instead of expansion.** Search BM25 twice, with the original question and with the expanded one, and fuse the two lists with `rrf()` before the weighted fusion. Does wine reach first place? What does it cost on the lookups?
4. **Grow the gazetteer.** Add Toronto as a city. Which question changes, and which kind of question could the same change break? (Look at "Can I work remotely from Italy for a few weeks?" and write one like it about a city.)
5. **Decompose a two-part question.** Split "My phone was stolen on a business trip. Who should I tell?" into "What counts as a security incident?" and "How do I report a security incident?", retrieve for each, and merge the contexts. Is How to report in it now? When would you rather do this than Day 11's surrounding-section retrieval?